In [37]:
import numpy as np
import pandas as pd

In [38]:
bioinfo = pd.read_parquet("../data/raw/bioinfo.parquet")
ideology = pd.read_parquet("../data/raw/ideology.parquet")
terms = pd.read_parquet("../data/raw/terms.parquet")

bill_summaries_119 = pd.read_parquet("../data/raw/bill_summaries_119.parquet")
committee_members = pd.read_parquet("../data/raw/committee_members.parquet")

committees = pd.read_parquet("../data/raw/committees.parquet")
sponsored_legislation = pd.read_parquet("../data/raw/sponsored_legislation.parquet")

schedule_e_2026 = pd.read_parquet("../data/raw/schedule_e_2026.parquet")

vote_compare = pd.read_parquet("../data/raw/vote_compare.parquet")

In [39]:
bioinfo = pd.read_parquet("../data/raw/bioinfo.parquet")
bioinfo.info()

<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bioguide_id     555 non-null    str    
 1   Full name       555 non-null    str    
 2   Chamber         555 non-null    str    
 3   State           555 non-null    str    
 4   Party           555 non-null    str    
 5   District        439 non-null    float64
 6   birthYear       554 non-null    str    
 7   image           554 non-null    str    
 8   Office address  539 non-null    str    
 9   Phone           539 non-null    str    
 10  Website         539 non-null    str    
dtypes: float64(1), str(10)
memory usage: 166.9 KB


In [40]:
bioinfo['bioguide_id'].duplicated().value_counts()

bioguide_id
False    555
Name: count, dtype: int64

In [41]:
bioinfo.head(3).T

,0,1,2
bioguide_id,R000575,S001185,P000609
Full name,Mike Rogers,Terri A. Sewell,Gary J. Palmer
Chamber,House of Representatives,House of Representatives,House of Representatives
State,Alabama,Alabama,Alabama
Party,Republican,Democratic,Republican
District,3.0,7.0,6.0
birthYear,1958,1965,1954
image,https://www.congress.gov/img/member/116_rp_al_...,https://www.congress.gov/img/member/s001185_20...,https://www.congress.gov/img/member/p000609_20...
Office address,"2469 Rayburn House Office Building, Washington...","1035 Longworth House Office Building, Washingt...","170 Cannon House Office Building, Washington, ..."
Phone,(202) 225-3261,(202) 225-2665,(202) 225-4921


In [42]:
def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.
    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.
    Returns:
        A DBML string representing the DataFrame schema.
    """
    dbml_string = f"Table {table_name} {{\n"
    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"
    dbml_string += "}\n"
    return dbml_string
def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [43]:
print(pandas_df_to_dbml(bioinfo, "members"))

Table members {
  bioguide_id varchar
  Full name varchar
  Chamber varchar
  State varchar
  Party varchar
  District float
  birthYear varchar
  image varchar
  Office address varchar
  Phone varchar
  Website varchar
}



In [44]:
bioinfo.columns = [x.replace(" ", "_").lower() for x in bioinfo.columns]
print(pandas_df_to_dbml(bioinfo, "members"))

Table members {
  bioguide_id varchar
  full_name varchar
  chamber varchar
  state varchar
  party varchar
  district float
  birthyear varchar
  image varchar
  office_address varchar
  phone varchar
  website varchar
}



In [45]:
ideology = pd.read_parquet("../data/raw/ideology.parquet")
ideology.info()

<class 'pandas.DataFrame'>
RangeIndex: 557 entries, 0 to 556
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   bioname              557 non-null    str    
 1   chamber              557 non-null    str    
 2   left_right_ideology  555 non-null    float64
 3   state_abbrev         557 non-null    str    
 4   district_code        557 non-null    int64  
 5   icpsr                557 non-null    int64  
 6   bioguide_id          556 non-null    str    
 7   party                557 non-null    str    
 8   fec_id               538 non-null    object 
dtypes: float64(1), int64(2), object(1), str(5)
memory usage: 60.7+ KB


In [46]:
ideology.head(13).T

,0,1,2,3,4,5,6,7,8,9,10,11,12
bioname,"TRUMP, Donald John","ROGERS, Mike Dennis","SEWELL, Terri","PALMER, Gary James","MOORE, Barry","STRONG, Dale","FIGURES, Shomari","ADERHOLT, Robert","BEGICH, Nicholas J., III","RADEWAGEN, Aumua Amata Coleman","GRIJALVA, Raúl M.","GOSAR, Paul","SCHWEIKERT, David"
chamber,President,House,House,House,House,House,House,House,House,House,House,House,House
left_right_ideology,0.403,0.379,-0.402,0.67,0.646,0.61,-0.411,0.405,0.59,0.327,-0.598,0.701,0.553
state_abbrev,USA,AL,AL,AL,AL,AL,AL,AL,AK,AS,AZ,AZ,AZ
district_code,0,3,7,6,1,5,2,4,1,0,7,9,1
icpsr,99912,20301,21102,21500,22140,22366,22515,29701,22503,21995,20305,21103,21105
bioguide_id,NaN,R000575,S001185,P000609,M001212,S001220,F000481,A000055,B001323,R000600,G000551,G000565,S001183
party,Republican,Republican,Democrat,Republican,Republican,Republican,Democrat,Republican,Republican,Republican,Democrat,Republican,Republican
fec_id,None,[H2AL03032],[H0AL07086],[H4AL06098],"[H8AL02171, S6AL00476]",[H2AL05102],[H4AL02170],[H6AL04098],[H2AK01083],[H4AS00036],None,[H0AZ01259],"[H4AZ06045, H8AZ05097]"


In [47]:
bioinfo.head(3).T

,0,1,2
bioguide_id,R000575,S001185,P000609
full_name,Mike Rogers,Terri A. Sewell,Gary J. Palmer
chamber,House of Representatives,House of Representatives,House of Representatives
state,Alabama,Alabama,Alabama
party,Republican,Democratic,Republican
district,3.0,7.0,6.0
birthyear,1958,1965,1954
image,https://www.congress.gov/img/member/116_rp_al_...,https://www.congress.gov/img/member/s001185_20...,https://www.congress.gov/img/member/p000609_20...
office_address,"2469 Rayburn House Office Building, Washington...","1035 Longworth House Office Building, Washingt...","170 Cannon House Office Building, Washington, ..."
phone,(202) 225-3261,(202) 225-2665,(202) 225-4921


In [48]:
ideology = ideology[['icpsr', 'bioguide_id', 'fec_id', 'left_right_ideology']]
ideology

,icpsr,bioguide_id,fec_id,left_right_ideology
0,99912,NaN,None,0.403
1,20301,R000575,[H2AL03032],0.379
2,21102,S001185,[H0AL07086],-0.402
3,21500,P000609,[H4AL06098],0.670
4,22140,M001212,"[H8AL02171, S6AL00476]",0.646
...,...,...,...,...
552,42504,J000312,[S4WV00332],0.562
553,29940,B001230,"[H8WI00018, S2WI00219]",-0.484
554,41111,J000293,[S0WI00197],0.648
555,20953,L000571,"[H8WY00148, S0WY00137]",0.682


In [49]:
fec = ideology.explode('fec_id')
fec

,icpsr,bioguide_id,fec_id,left_right_ideology
0,99912,NaN,NaN,0.403
1,20301,R000575,H2AL03032,0.379
2,21102,S001185,H0AL07086,-0.402
3,21500,P000609,H4AL06098,0.670
4,22140,M001212,H8AL02171,0.646
...,...,...,...,...
553,29940,B001230,S2WI00219,-0.484
554,41111,J000293,S0WI00197,0.648
555,20953,L000571,H8WY00148,0.682
555,20953,L000571,S0WY00137,0.682


In [50]:
ideology = ideology.drop(['fec_id'], axis=1)
fec = fec.drop(['left_right_ideology'], axis=1)

In [51]:
fec

,icpsr,bioguide_id,fec_id
0,99912,NaN,NaN
1,20301,R000575,H2AL03032
2,21102,S001185,H0AL07086
3,21500,P000609,H4AL06098
4,22140,M001212,H8AL02171
...,...,...,...
553,29940,B001230,S2WI00219
554,41111,J000293,S0WI00197
555,20953,L000571,H8WY00148
555,20953,L000571,S0WY00137


```
members = pd.merge(bioinfo, 
                   ideology,
                   how='outer',
                   #validate='one_to_one',
                   indicator='matched')
members['matched'].value_counts()

---------------------------------------------------------------------------
MergeError                                Traceback (most recent call last)
Cell In[47], line 1
----> 1 members = pd.merge(bioinfo, 
      2                    ideology,
      3                    how='outer',
      4                    validate='one_to_one',

File ~\anaconda3\envs\ds6600\Lib\site-packages\pandas\core\reshape\merge.py:385, in merge(left, right, how, on, left_on, right_on, left_index, right_index, sort, suffixes, copy, indicator, validate)
    371     return _cross_merge(
    372         left_df,
    373         right_df,
   (...)    382         validate=validate,
    383     )
    384 else:
--> 385     op = _MergeOperation(
    386         left_df,
    387         right_df,
    388         how=how,
    389         on=on,
    390         left_on=left_on,
    391         right_on=right_on,
    392         left_index=left_index,
    393         right_index=right_index,
    394         sort=sort,
...

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
 bioguide_id
    K000401 ...
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
```

In [52]:
ideology[ideology['bioguide_id'].duplicated()]

,icpsr,bioguide_id,left_right_ideology
78,92336,K000401,0.152


In [53]:
ideology.query('bioguide_id == "K000401"')

,icpsr,bioguide_id,left_right_ideology
60,22336,K000401,0.253
78,92336,K000401,0.152


In [54]:
ideology = ideology.groupby(['bioguide_id']).agg({'left_right_ideology': 'mean'}).reset_index() # average the duplicate congressmen's ideology scores to handle the duplicate row

In [55]:
members = pd.merge(bioinfo, 
                   ideology,
                   how='outer',
                   validate='one_to_one',
                   indicator='matched')
members['matched'].value_counts()
members = members.drop(['matched'], axis=1)

In [56]:
members.head(3).T

,0,1,2
bioguide_id,A000055,A000148,A000369
full_name,Robert B. Aderholt,Jake Auchincloss,Mark E. Amodei
chamber,House of Representatives,House of Representatives,House of Representatives
state,Alabama,Massachusetts,Nevada
party,Republican,Democratic,Republican
district,4.0,4.0,2.0
birthyear,1965,1988,1958
image,https://www.congress.gov/img/member/a000055_20...,https://www.congress.gov/img/member/67817e391f...,https://www.congress.gov/img/member/a000369_20...
office_address,"272 Cannon House Office Building, Washington, ...","1524 Longworth House Office Building, Washingt...","104 Cannon House Office Building, Washington, ..."
phone,(202) 225-4876,(202) 225-5931,(202) 225-6155


In [57]:
print(pandas_df_to_dbml(members, 'members'))

Table members {
  bioguide_id varchar
  full_name varchar
  chamber varchar
  state varchar
  party varchar
  district float
  birthyear varchar
  image varchar
  office_address varchar
  phone varchar
  website varchar
  left_right_ideology float
}



In [58]:
print(pandas_df_to_dbml(fec, 'crosswalk'))

Table crosswalk {
  icpsr int
  bioguide_id varchar
  fec_id varchar
}



In [59]:
terms = pd.read_parquet("../data/raw/terms.parquet")
terms.info()

<class 'pandas.DataFrame'>
RangeIndex: 3267 entries, 0 to 3266
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   bioguide_id  3267 non-null   str    
 1   chamber      3267 non-null   str    
 2   congress     3267 non-null   int64  
 3   stateCode    3267 non-null   str    
 4   startYear    3267 non-null   int64  
 5   endYear      3267 non-null   int64  
 6   district     2523 non-null   float64
dtypes: float64(1), int64(3), str(3)
memory usage: 272.4 KB


In [60]:
terms

,bioguide_id,chamber,congress,stateCode,startYear,endYear,district
0,R000575,House of Representatives,108,AL,2003,2005,3.0
1,R000575,House of Representatives,109,AL,2005,2007,3.0
2,R000575,House of Representatives,110,AL,2007,2009,3.0
3,R000575,House of Representatives,111,AL,2009,2011,3.0
4,R000575,House of Representatives,112,AL,2011,2013,3.0
...,...,...,...,...,...,...,...
3262,B001261,Senate,115,WY,2017,2019,NaN
3263,B001261,Senate,116,WY,2019,2021,NaN
3264,B001261,Senate,117,WY,2021,2023,NaN
3265,B001261,Senate,118,WY,2023,2025,NaN


In [61]:
# drop startyear and endyear
terms = terms.drop(['startYear', 'endYear'], axis=1)
terms.columns = [x.lower() for x in terms.columns]
print(pandas_df_to_dbml(terms, 'terms'))

Table terms {
  bioguide_id varchar
  chamber varchar
  congress int
  statecode varchar
  district float
}



In [62]:
# saving the processed dataframes to CSV files
members.to_csv("../data/processed/members.csv", index=False)
fec.to_csv("../data/processed/fec.csv", index=False)
terms.to_csv("../data/processed/terms.csv", index=False)

## Bill summaries and sponsored legistation

In [63]:
bill_summaries_119.head(3).T

,0,1,2
actionDate,2025-01-03,2025-01-09,2025-01-06
actionDesc,Introduced in House,Introduced in House,Introduced in Senate
currentChamber,House,House,Senate
currentChamberCode,H,H,S
lastSummaryUpdateDate,2025-01-09T01:58:40Z,2025-01-13T12:49:11Z,2025-01-13T16:21:37Z
text,<p><strong>Illegitimate Court Counteraction Ac...,<p>This resolution disapproves of the Central ...,<p><strong>Laken Riley Act</strong></p><p>This...
updateDate,2025-01-09T01:58:53Z,2025-01-13T12:51:00Z,2025-01-13T16:21:43Z
versionCode,00,00,00
bill.congress,119,119,119
bill.number,23,27,5


In [64]:
sponsored_legislation.head(3).T

,0,1,2
introducedDate,2026-05-13,2026-07-22,2026-07-22
number,8800,NaN,NaN
title,National Defense Authorization Act for Fiscal ...,NaN,NaN
type,HR,NaN,NaN
url,https://api.congress.gov/v3/bill/119/hr/8800?f...,https://api.congress.gov/v3/amendment/119/hamd...,https://api.congress.gov/v3/amendment/119/hamd...
bioguide_id,R000575,R000575,R000575


In [65]:
from bs4 import BeautifulSoup
mysoup = BeautifulSoup(bill_summaries_119['text'][0], 'html.parser')
mysoup.text

'Illegitimate Court Counteraction ActThis bill imposes sanctions against foreign persons (individuals and entities) who assist the International Criminal Court (ICC) in investigating, arresting, detaining, or prosecuting certain individuals.The bill categorizes as protected persons (1) any U.S.\xa0individual, U.S. entity, or person in the United States, unless the United States is a state party to the Rome Statute of the ICC and provides formal consent to ICC jurisdiction; and (2) any foreign person that is a citizen or lawful resident of a U.S. ally that is not a state party to the Rome Statute or has not consented to ICC jurisdiction.If the ICC attempts to investigate, arrest, detain or prosecute a protected person, the President must impose visa- and property-blocking sanctions against the foreign persons that engaged in or materially assisted in such actions, as well as against foreign persons owned by, controlled by, or acting on behalf of such foreign persons. The President must 

In [66]:
bill_summaries_119.shape

(6146, 16)

In [67]:
sponsored_legislation.shape

(26610, 6)

In [68]:
sponsored_legislation['url'].duplicated().value_counts()

url
False    26600
True        10
Name: count, dtype: int64

In [69]:
sponsored_legislation[sponsored_legislation['url'].duplicated(keep=False)]

,introducedDate,number,title,type,url,bioguide_id
13937,2025-04-04,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,S001150
13938,2025-04-04,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,S001150
15083,2026-02-12,3872,A bill to amend title XVIII of the Social Secu...,S,https://api.congress.gov/v3/bill/119/s/3872?fo...,B001277
15085,2026-02-12,3872,A bill to amend title XVIII of the Social Secu...,S,https://api.congress.gov/v3/bill/119/s/3872?fo...,B001277
20372,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20373,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20374,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20375,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20379,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20380,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608


In [70]:
[print(x) for x in sponsored_legislation[sponsored_legislation['url'].duplicated(keep=False)]['url']]

https://api.congress.gov/v3/amendment/119/samdt/2085?format=json
https://api.congress.gov/v3/amendment/119/samdt/2085?format=json
https://api.congress.gov/v3/bill/119/s/3872?format=json
https://api.congress.gov/v3/bill/119/s/3872?format=json
https://api.congress.gov/v3/amendment/119/samdt/842?format=json
https://api.congress.gov/v3/amendment/119/samdt/841?format=json
https://api.congress.gov/v3/amendment/119/samdt/840?format=json
https://api.congress.gov/v3/amendment/119/samdt/839?format=json
https://api.congress.gov/v3/amendment/119/samdt/842?format=json
https://api.congress.gov/v3/amendment/119/samdt/841?format=json
https://api.congress.gov/v3/amendment/119/samdt/840?format=json
https://api.congress.gov/v3/amendment/119/samdt/839?format=json
https://api.congress.gov/v3/amendment/119/samdt/900?format=json
https://api.congress.gov/v3/amendment/119/samdt/900?format=json
https://api.congress.gov/v3/amendment/119/samdt/2541?format=json
https://api.congress.gov/v3/amendment/119/samdt/2542?

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [71]:
bill_summaries_119[bill_summaries_119['bill.url'].duplicated(keep=False)]

,actionDate,actionDesc,currentChamber,currentChamberCode,lastSummaryUpdateDate,text,updateDate,versionCode,bill.congress,bill.number,bill.originChamber,bill.originChamberCode,bill.title,bill.type,bill.updateDateIncludingText,bill.url
2,2025-01-06,Introduced in Senate,Senate,S,2025-01-13T16:21:37Z,<p><strong>Laken Riley Act</strong></p><p>This...,2025-01-13T16:21:43Z,00,119,5,Senate,S,Laken Riley Act,S,2026-07-30,https://api.congress.gov/v3/bill/119/s/5?forma...
44,2025-01-16,Introduced in House,House,H,2025-01-27T11:42:14Z,<p>This joint resolution nullifies the rule ti...,2025-01-27T11:44:23Z,00,119,24,House,H,Providing for congressional disapproval under ...,HJRES,2026-07-13,https://api.congress.gov/v3/bill/119/hjres/24?...
45,2025-01-15,Introduced in House,House,H,2025-01-27T14:02:08Z,<p>This joint resolution nullifies the rule ti...,2025-01-27T14:02:43Z,00,119,20,House,H,Providing for congressional disapproval under ...,HJRES,2026-09-19,https://api.congress.gov/v3/bill/119/hjres/20?...
54,2025-01-03,Introduced in House,House,H,2025-01-30T16:54:11Z,<p><b>Wounded Knee Massacre Memorial and Sacre...,2025-01-30T16:54:15Z,00,119,165,House,H,Wounded Knee Massacre Memorial and Sacred Site...,HR,2026-07-13,https://api.congress.gov/v3/bill/119/hr/165?fo...
80,2025-01-03,Introduced in House,House,H,2025-02-03T14:54:09Z,<p><strong>Alaska Native Settlement Trust Elig...,2025-02-03T14:56:55Z,00,119,42,House,H,Alaska Native Settlement Trust Eligibility Act,HR,2026-07-13,https://api.congress.gov/v3/bill/119/hr/42?for...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5815,2026-09-16,Public Law,Unknown,Unknown,2026-09-18T19:00:10Z,<p><strong>Prison Staff Safety Enhancement Act...,2026-09-18T19:00:49Z,49,119,307,Senate,S,Prison Staff Safety Enhancement Act,S,2026-09-21,https://api.congress.gov/v3/bill/119/s/307?for...
5816,2026-09-11,Public Law,Unknown,Unknown,2026-09-18T19:07:13Z,<p><strong>Doug LaMalfa Federal Disaster Tax R...,2026-09-18T19:12:06Z,49,119,5366,House,H,Doug LaMalfa Federal Disaster Tax Relief Certa...,HR,2026-09-21,https://api.congress.gov/v3/bill/119/hr/5366?f...
5823,2026-06-24,Reported to Senate,Senate,S,2026-09-21T20:53:33Z,<p><strong>Protect College Sports Act of 2026<...,2026-09-21T20:54:12Z,25,119,4668,Senate,S,Protect College Sports Act of 2026,S,2026-10-07,https://api.congress.gov/v3/bill/119/s/4668?fo...
5917,2026-09-15,Passed House,House,H,2026-09-28T14:52:37Z,<p><strong>Protecting Taxpayers from Ghost Pre...,2026-09-28T14:57:21Z,53,119,9499,House,H,Protecting Taxpayers from Ghost Preparers Act,HR,2026-09-28,https://api.congress.gov/v3/bill/119/hr/9499?f...


In [72]:
sponsored_legislation = sponsored_legislation.drop_duplicates() # remove duplicate rows based on all columns
sponsored_legislation = sponsored_legislation[['bioguide_id', 'url']] # remove all columns except bioguide_id and url

bills = pd.merge(bill_summaries_119, sponsored_legislation, left_on = ['bill.url'], right_on = ['url'], how = 'outer', indicator = 'matched', validate = 'many_to_one')

In [73]:
bills['matched'].value_counts()
# the reason that there are 2,000+ unmatched rows is because the bill_summaries_119 dataset contains bills from the 119th congress, 
# # while the sponsored_legislation dataset contains bills from further back 
# Therefore, there are many bills in the bill_summaries_119 dataset that do not have a corresponding entry in the sponsored_legislation dataset.

matched
right_only    20597
both           6146
left_only         0
Name: count, dtype: int64

In [74]:
bills = bills.query('matched == "both"')
bills = bills.drop(['matched'], axis=1)
bills.columns = [x.lower().replace('.', '_') for x in bills.columns]
bills

,actiondate,actiondesc,currentchamber,currentchambercode,lastsummaryupdatedate,text,updatedate,versioncode,bill_congress,bill_number,bill_originchamber,bill_originchambercode,bill_title,bill_type,bill_updatedateincludingtext,bill_url,bioguide_id,url
7021,2026-05-13,Introduced in House,House,H,2026-06-24T14:07:00Z,<p>This concurrent resolution directs the Pres...,2026-06-24T14:07:20Z,00,119.0,100,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,F000477,https://api.congress.gov/v3/bill/119/hconres/1...
7022,2026-05-14,Introduced in House,House,H,2026-07-18T16:28:55Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:00Z,00,119.0,101,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,J000305,https://api.congress.gov/v3/bill/119/hconres/1...
7023,2026-05-15,Introduced in House,House,H,2026-07-17T20:36:05Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:10Z,00,119.0,102,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,R000621,https://api.congress.gov/v3/bill/119/hconres/1...
7024,2026-05-20,Introduced in House,House,H,2026-07-18T16:29:53Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:58Z,00,119.0,103,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-22,https://api.congress.gov/v3/bill/119/hconres/1...,A000380,https://api.congress.gov/v3/bill/119/hconres/1...
7025,2026-05-20,Introduced in House,House,H,2026-07-17T20:36:47Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:52Z,00,119.0,104,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-07-17,https://api.congress.gov/v3/bill/119/hconres/1...,B001313,https://api.congress.gov/v3/bill/119/hconres/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26737,2025-02-25,Introduced in Senate,Senate,S,2026-09-23T16:14:06Z,<p>This resolution designates February 23-Marc...,2026-09-23T16:14:34Z,00,119.0,95,Senate,S,A resolution expressing support for the design...,SRES,2026-09-23,https://api.congress.gov/v3/bill/119/sres/95?f...,S001217,https://api.congress.gov/v3/bill/119/sres/95?f...
26738,2025-02-25,Introduced in Senate,Senate,S,2025-05-07T19:53:39Z,<p>This resolution designates February 24-Febr...,2025-05-07T19:54:03Z,00,119.0,96,Senate,S,A resolution designating the week of February ...,SRES,2025-07-01,https://api.congress.gov/v3/bill/119/sres/96?f...,C001035,https://api.congress.gov/v3/bill/119/sres/96?f...
26740,2025-02-26,Introduced in Senate,Senate,S,2025-05-08T19:34:05Z,<p>This resolution condemns China's Hong Kong ...,2025-05-08T19:34:25Z,00,119.0,98,Senate,S,A resolution condemning Beijing's destruction ...,SRES,2025-05-08,https://api.congress.gov/v3/bill/119/sres/98?f...,R000584,https://api.congress.gov/v3/bill/119/sres/98?f...
26741,2025-02-26,Introduced in Senate,Senate,S,2025-09-19T20:56:40Z,<p>This resolution recognizes Black History Mo...,2025-09-19T20:57:04Z,00,119.0,99,Senate,S,A resolution celebrating Black History Month.,SRES,2026-03-03,https://api.congress.gov/v3/bill/119/sres/99?f...,B001288,https://api.congress.gov/v3/bill/119/sres/99?f...


In [75]:
bills.duplicated(['bill_url', 'versioncode']).value_counts()

False    6146
Name: count, dtype: int64

In [76]:
bills = bills.drop('url', axis=1)
bills.to_csv("../data/processed/bills.csv", index=False)

In [78]:
print(pandas_df_to_dbml(bills, 'bills'))

Table bills {
  actiondate varchar
  actiondesc varchar
  currentchamber varchar
  currentchambercode varchar
  lastsummaryupdatedate varchar
  text varchar
  updatedate varchar
  versioncode varchar
  bill_congress float
  bill_number varchar
  bill_originchamber varchar
  bill_originchambercode varchar
  bill_title varchar
  bill_type varchar
  bill_updatedateincludingtext varchar
  bill_url varchar
  bioguide_id varchar
}



## FEC Schedule E

In [81]:
pd.set_option('display.max_columns', 120)
schedule_e_2026.head(3).T

,0,1,2
fec_id,H4AL06098,H4AL06098,H8AL02171
bioguide_id,P000609,P000609,M001212
action_code,A,A,A
action_code_full,ADD,ADD,ADD
amendment_indicator,N,N,N
...,...,...,...
committee.state_full,District Of Columbia,District Of Columbia,Wisconsin
committee.street_1,"700 2ND STREET, NE","700 2ND STREET, NE",PO BOX 183
committee.street_2,NaN,NaN,NaN
committee.treasurer_name,NaN,NaN,"DATWYLER, THOMAS"


In [82]:
schedule_e_2026.head(3)

,fec_id,bioguide_id,action_code,action_code_full,amendment_indicator,amendment_number,back_reference_schedule_name,back_reference_transaction_id,candidate_first_name,candidate_id,candidate_last_name,candidate_middle_name,candidate_name,candidate_office,candidate_office_district,candidate_office_state,candidate_party,candidate_prefix,candidate_suffix,category_code,category_code_full,committee_id,conduit_committee_city,conduit_committee_id,conduit_committee_name,conduit_committee_state,conduit_committee_street1,conduit_committee_street2,conduit_committee_zip,disbursement_dt,dissemination_date,election_type,election_type_full,expenditure_amount,expenditure_date,expenditure_description,file_number,filer_first_name,filer_last_name,filer_middle_name,filer_prefix,filer_suffix,filing_date,filing_form,form_line_number,image_number,independent_sign_date,independent_sign_name,is_notice,line_number,link_id,memo_code,memo_code_full,memo_text,memoed_subtotal,most_recent,notary_commission_expiration_date,notary_sign_date,notary_sign_name,office_total_ytd,...,payee_street_1,payee_street_2,payee_suffix,payee_zip,pdf_url,previous_file_number,report_type,report_year,schedule_type,schedule_type_full,sub_id,support_oppose_indicator,transaction_id,candidate.candidate_id,candidate.idx,candidate.two_year_period,committee.affiliated_committee_name,committee.candidate_ids,committee.city,committee.committee_id,committee.committee_type,committee.committee_type_full,committee.cycle,committee.cycles,committee.cycles_has_activity,committee.cycles_has_financial,committee.designated_agent_city,committee.designated_agent_first_name,committee.designated_agent_last_name,committee.designated_agent_middle_name,committee.designated_agent_name,committee.designated_agent_phone_number,committee.designated_agent_prefix,committee.designated_agent_state,committee.designated_agent_street1,committee.designated_agent_street2,committee.designated_agent_suffix,committee.designated_agent_title,committee.designated_agent_zip,committee.designation,committee.designation_full,committee.filing_frequency,committee.first_f1_date,committee.first_file_date,committee.is_active,committee.last_cycle_has_activity,committee.last_cycle_has_financial,committee.last_f1_date,committee.last_file_date,committee.name,committee.organization_type,committee.organization_type_full,committee.party,committee.party_full,committee.state,committee.state_full,committee.street_1,committee.street_2,committee.treasurer_name,committee.zip
0,H4AL06098,P000609,A,ADD,N,0.0,NaN,NaN,GARY,H4AL06098,PALMER,NaN,"PALMER, GARY",H,06,AL,REP,NaN,NaN,004,Advertising Expenses,C90011578,None,NaN,None,None,None,None,None,2026-05-05,NaN,P2026,PRIMARY,131018.0,2026-05-05,"TELEVISION AND ONLINE ADVERTISEMENT, INDEPENDE...",1991845.0,NaN,NaN,NaN,NaN,NaN,2026-07-14,F5,NaN,202607149874973279,NaN,NaN,False,NaN,4.071420e+18,NaN,None,NaN,False,True,None,None,None,131018.0,...,107 S. WEST STREET,#506,NaN,22314,https://docquery.fec.gov/cgi-bin/fecimg/?20260...,1991845.0,Q2,2026,SE-F57,FOR EACH INDEPENDENT EXPENDITURE MADE,4072820261538847022,S,F57.4141,H4AL06098,42153.0,2026,NaN,[],WASHINGTON,C90011578,I,Independent expenditure filer (not a committee),2026.0,"[2010, 2012, 2014, 2016, 2018, 2020, 2022, 202...","[2010, 2012, 2014, 2018, 2024, 2026]","[2010, 2012, 2014, 2018, 2026]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,U,Unauthorized,Q,NaN,2010-08-10,True,2026.0,2026.0,NaN,2026-08-21,"AMERICAN CHEMISTRY COUNCIL, INC",NaN,NaN,NaN,NaN,DC,District Of Columbia,"700 2ND STREET, NE",NaN,NaN,20002
1,H4AL06098,P000609,A,ADD,N,NaN,NaN,NaN,GARY,H4AL06098,PALMER,NaN,"PALMER, GARY",H,06,AL,REP,NaN,NaN,004,Advertising Expenses,C90011578,None,NaN,None,None,None,None,None,2026-05-05,NaN,P2026,PRIMARY,131018.0,2026-05-05,"TELEVISION AND ONLINE ADVERTISEMENT, INDEPENDE...",1972927.0,NaN,NaN,NaN,NaN,NaN,2026-05-06,F5,NaN,202605069866842098,NaN,NaN,True,NaN,4.050620e+18,NaN,None,NaN,False,True,None,None,None,131018.0,...,107 S. WEST STREET,#50

In [ ]:
schedule_e_2026.columns

# -- what do we want to know? -- 
# who was the expenditure made for? fec_id
# how much was the expenditure? expenditure_amount
# who made the expenditure? committee.name
# was it a support or attack ad? support_oppose_indicator
# super PAC or independent expenditure committee? committee.committee_type_full

Index(['fec_id', 'bioguide_id', 'action_code', 'action_code_full',
       'amendment_indicator', 'amendment_number',
       'back_reference_schedule_name', 'back_reference_transaction_id',
       'candidate_first_name', 'candidate_id',
       ...
       'committee.organization_type', 'committee.organization_type_full',
       'committee.party', 'committee.party_full', 'committee.state',
       'committee.state_full', 'committee.street_1', 'committee.street_2',
       'committee.treasurer_name', 'committee.zip'],
      dtype='str', length=128)

In [88]:
expenditures = schedule_e_2026[['fec_id', 'transaction_id', 'expenditure_amount', 'committee.name', 'committee.committee_type_full', 'support_oppose_indicator']]
expenditures

,fec_id,transaction_id,expenditure_amount,committee.name,committee.committee_type_full,support_oppose_indicator
0,H4AL06098,F57.4141,131018.00,"AMERICAN CHEMISTRY COUNCIL, INC",Independent expenditure filer (not a committee),S
1,H4AL06098,F57.4143,131018.00,"AMERICAN CHEMISTRY COUNCIL, INC",Independent expenditure filer (not a committee),S
2,H8AL02171,E0E0A659AE9C94E6CA52,15000.00,CONSTITUTIONAL CONSERVATIVES FUND,Super PAC (Independent Expenditure-Only),S
3,H8AL02171,E0E0A659AE9C94E6CA52,15000.00,CONSTITUTIONAL CONSERVATIVES FUND,Super PAC (Independent Expenditure-Only),S
4,H8AL02171,E77F981314DDE44B285A,109.20,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
...,...,...,...,...,...,...
15767,S0WY00137,E19CD48FA103A4B08818,53.77,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15768,S0WY00137,E85960C39314B40C189C,103.42,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15769,S0WY00137,EFF4917D0897E4FB193F,25.42,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15770,S0WY00137,E84341A51E0FF43EE934,26.75,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S


In [91]:
expenditures = expenditures.rename({'committee.name': 'committee_name', 'committee.committee_type_full': 'committee_type_full'}, axis=1)
expenditures

,fec_id,transaction_id,expenditure_amount,committee_name,committee_type_full,support_oppose_indicator
0,H4AL06098,F57.4141,131018.00,"AMERICAN CHEMISTRY COUNCIL, INC",Independent expenditure filer (not a committee),S
1,H4AL06098,F57.4143,131018.00,"AMERICAN CHEMISTRY COUNCIL, INC",Independent expenditure filer (not a committee),S
2,H8AL02171,E0E0A659AE9C94E6CA52,15000.00,CONSTITUTIONAL CONSERVATIVES FUND,Super PAC (Independent Expenditure-Only),S
3,H8AL02171,E0E0A659AE9C94E6CA52,15000.00,CONSTITUTIONAL CONSERVATIVES FUND,Super PAC (Independent Expenditure-Only),S
4,H8AL02171,E77F981314DDE44B285A,109.20,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
...,...,...,...,...,...,...
15767,S0WY00137,E19CD48FA103A4B08818,53.77,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15768,S0WY00137,E85960C39314B40C189C,103.42,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15769,S0WY00137,EFF4917D0897E4FB193F,25.42,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15770,S0WY00137,E84341A51E0FF43EE934,26.75,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S


In [92]:
print(pandas_df_to_dbml(expenditures, 'expenditures'))

Table expenditures {
  fec_id varchar
  transaction_id varchar
  expenditure_amount float
  committee_name varchar
  committee_type_full varchar
  support_oppose_indicator varchar
}



In [95]:
expenditures.to_csv("../data/processed/expenditures.csv", index=False)